# LoRA Training & Inference Cost Benchmark — single-hardware run

Run **one** hardware target end-to-end: build dataset → LoRA SFT → vLLM serve + benchmark → compare against the bundled reference results. Set `HARDWARE` to `A100`, `H100`, or `B200` to match your workspace's GPUs.

> The full measured sweep across all three platforms lives in `../results/` and `../benchmarks.md`. This notebook reproduces one slice; the analysis at the end compares your run to the bundled reference.


---
## 1. Setup

`google/gemma-4-31B-it` is gated — set `HF_TOKEN`. Pick your hardware target.


In [ ]:
import os
os.environ.setdefault("HARDWARE", "A100")   # A100 | H100 | B200
# os.environ["HF_TOKEN"] = "hf_..."          # required: gemma-4-31B-it is gated
!pip install -q -r ../requirements.txt


---
## 2. Build the packed ShareGPT workload

Downloads the public ShareGPT dataset and greedy-packs it to `seq_len=2048` (see `../data/DATASET_CARD.md`).


In [ ]:
!python ../data_prep/pack_sharegpt.py \
    --model-id google/gemma-4-31B-it \
    --out /shared/datasets/sharegpt_packed_2048


---
## 3. LoRA fine-tune (one variant)

`fsdp-nogc` — FSDP without gradient checkpointing, the cheapest training config on A100/H100. 8 GPUs via `torchrun`.


In [ ]:
HW = os.environ["HARDWARE"]
!torchrun --nproc_per_node=8 ../batch-job/train_sft.py \
    --hardware {HW} --variant fsdp-nogc --quant bf16 \
    --dataset /shared/datasets/sharegpt_packed_2048 \
    --out-dir /shared/runs/{HW}-fsdp-nogc


---
## 4. Serve + benchmark inference

Base model + MTP draft length γ=1 over a concurrency sweep, via vLLM (8 instances, TP=1).


In [ ]:
!HARDWARE={os.environ["HARDWARE"]} MTP_ON=1 MTP_GAMMA=1 CONCURRENCIES="16 64 256" \
    bash ../batch-job/serve_and_bench.sh


---
## 5. Compare against the bundled reference results

No GPU needed for this part — it reads the bundled measured sweep.


In [ ]:
import pandas as pd
R = "../results"
train = pd.read_csv(f"{R}/training.csv")
cost  = pd.read_csv(f"{R}/cost_summary.csv")

# Best (fastest) training config per hardware from the reference sweep:
best = (train[train.tokens_per_sec_aggregate.notna()]
        .sort_values("tokens_per_sec_aggregate", ascending=False)
        .groupby("hardware").head(1)[["hardware","variant","tokens_per_sec_aggregate","peak_vram_gib_rank0"]])
print("Best training config per platform (reference):")
print(best.to_string(index=False))

# Relative cost-per-token index (A100 = 100), inference peak:
cost[cost.scope == "inference_peak"][["hardware","tps_aggregate","relative_index_vs_A100"]]


---
## 6. Regenerate the figures

Rebuilds every figure in `../images/` from `../results/*.csv` (cost columns are relative indices, A100 = 100; no absolute \$).


In [ ]:
!python ../analysis/make_figures.py
from IPython.display import Image
Image("../images/fig1_training_tps.png")


---
## Done

You ran one hardware target and compared it to the bundled measured sweep. To benchmark another GPU, change `HARDWARE` and re-run, or use Path B: `./batch-job/submit.sh H100 my-run`.

See `../benchmarks.md` for the full results and `../README.md` for the findings and the [blog walkthrough](https://vessl.ai/en/blog/lora-finetuning-cost-a100-h100-b200).
